# Notebook 17 — Generate Supplementary Information figures (System I)

## Purpose

Regenerates every figure `docs/Manuscript/supporting_information.tex`'s System I
sections (S1-S6) cite, using the current, matched-protocol pipeline (per-draw
`warm_start_ic`, no fixed-healthy-warm-start mismatch — see `HANDOFF.md`'s Item 8
history). Three of the source `.png` files were missing from `figures/` entirely
(deleted in an earlier, unrelated legacy cleanup), and at least one
(`02_headline_summary.png`, "Figure 2") was flagged as looking outdated — this
notebook regenerates all of them from scratch rather than patching individually,
so every figure in the SI is traceable to one script run under one protocol.

**Depends on** `src/cstr_sbi/{physics,simulator,plotting}.py`, restored this session
from git history after an earlier cleanup deleted them from the working tree
(verified not to conflict with the separate `cstr_sbi.recycle`/`cstr_sbi.luyben`
System II subpackages).

## Output files (all written to `figures/`)

| Figure | File | SI section |
|---|---|---|
| 1 | `02_scenarios_overview.png` | S1 |
| 2 | `02_headline_summary.png` | S1 |
| 3 | `04_simulator_sanity.png` | S2 |
| 4 | `03_pca.png` | S4 |
| 5 | `04_sensitivity_scatter.png` | S3 |
| 6 | `04_sbc_ranks.png` | S6 |

Also (re)writes `data/observations.npz` and `data/scenario_configs.csv` (also
deleted in the same cleanup, and required by every figure below).


## Section 1 — Imports and setup


In [1]:
import time
from pathlib import Path

import jax
import jax.numpy as jnp
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from scipy.stats import gaussian_kde
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

from cstr_sbi.physics import (
    NOMINAL_INLET, NOMINAL_INLET_CL, NOMINAL_CTRL, NOMINAL_Y0, NOMINAL_PARAMS_CL,
    QC0, TSP, K0_NOMINAL, UA_NOMINAL,
)
from cstr_sbi.scenarios import SCENARIO_CONFIGS, list_configs
from cstr_sbi.simulator import (
    DEFAULT_PROCESS_SIGMA, DEFAULT_SENSOR_NOISE_PCT,
    DEFAULT_DT_INT, DEFAULT_DT_OUT,
    generate_replicates, warm_start_ic, simulate_em_window, apply_sensor_layer,
    _em_scan_open_loop,
)
from cstr_sbi.summaries import (
    FEATURE_NAMES, N_FEATURES, compute_summary_statistics, compute_summary_statistics_batch,
)
from cstr_sbi.inference import sample_posterior, train_sbi_posterior, simulation_wrapper_sbi
from cstr_sbi.priors import ALPHA_HIGH, ALPHA_LOW, BETA_HIGH, BETA_LOW, box_uniform_2d
from cstr_sbi.metrics import summarise_posterior

jax.config.update("jax_enable_x64", False)

ROOT = Path.cwd().parent.parent if (Path.cwd() / "system_I").exists() else Path.cwd().parent
if not (ROOT / "src").exists():
    ROOT = Path.cwd().parents[1]
DATA = ROOT / "data"; DATA.mkdir(exist_ok=True)
RESULTS = ROOT / "results"; RESULTS.mkdir(exist_ok=True)
FIGS = ROOT / "figures"; FIGS.mkdir(exist_ok=True)

print(f"ROOT={ROOT}")
print("JAX devices:", jax.devices())


ROOT=/Users/simo/inso-po-RD/cstr-model-optimisation/research_project
JAX devices: [CpuDevice(id=0)]


## Section 2 — Regenerate the System I dataset

Exact code from `notebooks/system_I/02_data_generation.ipynb` (already uses per-scenario `warm_start_ic` — this was never affected by the fixed-warm-start bug that affected SBI training/SBC evaluation data). 8 scenarios x 50 replicates = 400 windows.


In [2]:
N_REPLICATES = 50
T_WINDOW = 60.0

def generate_open_loop_replicates(
    UA_eff, k0_eff, n_replicates, master_key,
    t_window=T_WINDOW, dt=DEFAULT_DT_INT, dt_out=DEFAULT_DT_OUT,
    sigma_proc=DEFAULT_PROCESS_SIGMA, noise_pct=DEFAULT_SENSOR_NOISE_PCT,
    drift_T=0.0,
):
    # Open-loop variant -- 3-state, fixed Qc = QC0.
    params_ol = jnp.array([UA_eff, k0_eff])
    inlet_ol = jnp.asarray(NOMINAL_INLET).at[3].set(QC0)
    y0_ol = jnp.array([0.018, 312.15, 299.05])

    n_steps = int(round(t_window / dt))
    stride = int(round(dt_out / dt))
    keys = jax.random.split(master_key, 2 * n_replicates)
    proc_keys, sens_keys = keys[:n_replicates], keys[n_replicates:]

    @jax.vmap
    def one_rep(proc_k, sens_k):
        ys3 = _em_scan_open_loop(y0_ol, proc_k, params_ol, inlet_ol,
                                 dt, sigma_proc, n_steps, stride)
        qc_const = jnp.full((ys3.shape[0],), QC0)
        obs4 = jnp.stack([ys3[:, 0], ys3[:, 1], ys3[:, 2], qc_const], axis=1)
        return apply_sensor_layer(obs4, key=sens_k,
                                  noise_pct=noise_pct, drift_T=drift_T)
    obs = one_rep(proc_keys, sens_keys)
    t_out = jnp.arange(1, obs.shape[1] + 1) * dt_out
    return t_out, obs


master_key = jax.random.PRNGKey(2026)
all_results = {}

t0 = time.perf_counter()
for sc in list_configs():
    key = jax.random.fold_in(master_key, sc.id)
    if sc.mode == "closed_loop":
        params = sc.params()
        y0 = warm_start_ic(params, NOMINAL_INLET_CL, NOMINAL_CTRL)
        t_out, obs = generate_replicates(
            params, NOMINAL_INLET_CL, NOMINAL_CTRL, y0,
            n_replicates=N_REPLICATES, master_key=key,
            t_window=T_WINDOW, drift_T=sc.drift_T,
        )
    else:
        UA_eff = sc.beta * UA_NOMINAL
        k0_eff = sc.alpha * K0_NOMINAL
        t_out, obs = generate_open_loop_replicates(
            UA_eff, k0_eff, N_REPLICATES, key, drift_T=sc.drift_T,
        )
    obs_np = np.asarray(obs)
    all_results[sc.name] = {"t": np.asarray(t_out), "obs": obs_np, "config": sc}
    print(f"  {sc.name:28s}  {obs_np.shape}  "
          f"<C>={obs_np[:,:,0].mean():.4f}  <T>={obs_np[:,:,1].mean():.2f}  "
          f"<Qc>={obs_np[:,:,3].mean():.2f}")
print(f"\nTotal generation time: {time.perf_counter()-t0:.2f}s")

thetas, xs, ids, modes = [], [], [], []
for sc in list_configs():
    res = all_results[sc.name]
    n = res["obs"].shape[0]
    thetas.append(np.tile(np.asarray(sc.params()), (n, 1)))
    xs.append(res["obs"])
    ids.append(np.full(n, sc.id, dtype=np.int32))
    modes.append(np.array([sc.mode] * n))

theta = np.concatenate(thetas)
x = np.concatenate(xs, axis=0)
scenario_id = np.concatenate(ids)
mode = np.concatenate(modes)
t_grid = res["t"]

np.savez(DATA / "observations.npz", theta=theta, x=x, scenario_id=scenario_id, mode=mode, t=np.asarray(t_grid))
print(f"Saved {DATA / 'observations.npz'}")

scenario_rows = [
    {"id": sc.id, "name": sc.name, "mode": sc.mode, "alpha": sc.alpha, "beta": sc.beta,
     "drift_T": sc.drift_T, "description": sc.description}
    for sc in list_configs()
]
scenarios_df = pd.DataFrame(scenario_rows).set_index("id")
scenarios_df.to_csv(DATA / "scenario_configs.csv")
print(f"Saved {DATA / 'scenario_configs.csv'}")


  Sc0_open_healthy              (50, 120, 4)  <C>=0.0184  <T>=312.15  <Qc>=80.01


  Sc1_closed_healthy            (50, 120, 4)  <C>=0.0179  <T>=312.47  <Qc>=52.70
  Sc2_closed_fouling            (50, 120, 4)  <C>=0.0171  <T>=312.97  <Qc>=214.03
  Sc3_closed_decay              (50, 120, 4)  <C>=0.0254  <T>=312.42  <Qc>=45.77


  Sc4_closed_combined           (50, 120, 4)  <C>=0.0207  <T>=312.56  <Qc>=91.56


  Sc5_closed_saturated          (50, 120, 4)  <C>=0.0153  <T>=314.17  <Qc>=382.96
  Sc6_open_with_fault           (50, 120, 4)  <C>=0.0171  <T>=312.95  <Qc>=80.01
  Sc7_closed_drift              (50, 120, 4)  <C>=0.0176  <T>=314.58  <Qc>=101.24

Total generation time: 2.27s
Saved /Users/simo/inso-po-RD/cstr-model-optimisation/research_project/data/observations.npz
Saved /Users/simo/inso-po-RD/cstr-model-optimisation/research_project/data/scenario_configs.csv


## Section 3 — Figure 1: `02_scenarios_overview.png` (SI S1)


In [3]:
def plot_scenario(ax_row, name, n_show=6):
    res = all_results[name]
    t = res["t"]; obs = res["obs"]; sc = res["config"]
    labels = ["C [mol/L]", "T [K]", "Tc [K]", "Qc [L/min]"]
    for ax, lab, ch in zip(ax_row, labels, range(4)):
        for i in range(min(n_show, obs.shape[0])):
            ax.plot(t, obs[i, :, ch], lw=0.6, alpha=0.7)
        ax.plot(t, obs[:n_show, :, ch].mean(axis=0), color="k", lw=1.2, label="mean")
        ax.set_xlabel("t [min]"); ax.set_ylabel(lab); ax.grid(alpha=0.25)
    ax_row[0].text(
        -0.30, 0.5, f"{name}\n(alpha={sc.alpha}, beta={sc.beta})",
        transform=ax_row[0].transAxes, ha="right", va="center", fontsize=9,
    )

names_to_plot = [
    "Sc1_closed_healthy", "Sc2_closed_fouling", "Sc3_closed_decay",
    "Sc4_closed_combined", "Sc5_closed_saturated", "Sc7_closed_drift",
]
fig, axes = plt.subplots(len(names_to_plot), 4, figsize=(13, 2.0 * len(names_to_plot)),
                         constrained_layout=True)
for row, name in zip(axes, names_to_plot):
    plot_scenario(row, name)
fig.suptitle("Closed-loop scenarios -- 6 replicates per scenario, 4 observable channels", fontsize=12)
fig_path = FIGS / "02_scenarios_overview.png"
fig.savefig(fig_path, dpi=120)
plt.close(fig)
print(f"Saved: {fig_path}")


Saved: /Users/simo/inso-po-RD/cstr-model-optimisation/research_project/figures/02_scenarios_overview.png


## Section 4 — Figure 2: `02_headline_summary.png` (SI S1)


In [4]:
rows = []
for sc in list_configs():
    res = all_results[sc.name]
    obs = res["obs"]
    rows.append({
        "id": sc.id, "name": sc.name, "mode": sc.mode, "alpha": sc.alpha, "beta": sc.beta,
        "C_mean": obs[:,:,0].mean(), "T_mean": obs[:,:,1].mean(), "Tc_mean": obs[:,:,2].mean(),
        "Qc_mean": obs[:,:,3].mean(), "T_std": obs[:,:,1].std(), "Qc_std": obs[:,:,3].std(),
    })
summary = pd.DataFrame(rows).set_index("id")
print(summary.to_string(float_format=lambda v: f"{v:9.4f}"))

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4), constrained_layout=True)
order = summary.sort_values("id").index
ax1.bar(summary.loc[order, "name"], summary.loc[order, "T_mean"] - TSP,
        yerr=summary.loc[order, "T_std"], capsize=3, color="C0", alpha=0.85)
ax1.set_ylabel("T_mean - Tsp [K]")
ax1.set_title("Reactor temperature deviation from setpoint")
ax1.tick_params(axis="x", rotation=45)
ax1.grid(alpha=0.3, axis="y"); ax1.axhline(0, color="k", lw=0.6)

ax2.bar(summary.loc[order, "name"], summary.loc[order, "Qc_mean"],
        yerr=summary.loc[order, "Qc_std"], capsize=3, color="C2", alpha=0.85)
ax2.axhline(QC0, color="k", ls=":", lw=0.7, label=f"Qc0 = {QC0}")
ax2.axhline(400.0, color="C3", ls="--", lw=0.7, label="Qc_max = 400")
ax2.set_ylabel("Qc_mean [L/min]")
ax2.set_title("Coolant flow per scenario")
ax2.tick_params(axis="x", rotation=45)
ax2.grid(alpha=0.3, axis="y"); ax2.legend(fontsize=8)

fig.suptitle("Headline -- T regulated, Qc carries the fault signal", fontsize=12)
fig_path = FIGS / "02_headline_summary.png"
fig.savefig(fig_path, dpi=120)
plt.close(fig)
print(f"Saved: {fig_path}")


                    name         mode     alpha      beta    C_mean    T_mean   Tc_mean   Qc_mean     T_std    Qc_std
id                                                                                                                   
0       Sc0_open_healthy    open_loop    1.0000    1.0000    0.0184  312.1538  299.0718   80.0103    1.5996    0.4022
1     Sc1_closed_healthy  closed_loop    1.0000    1.0000    0.0179  312.4719  300.2291   52.6956    1.5497   20.8611
2     Sc2_closed_fouling  closed_loop    1.0000    0.7000    0.0171  312.9688  297.6511  214.0259    1.5646   30.5805
3       Sc3_closed_decay  closed_loop    0.7000    1.0000    0.0254  312.4171  300.6247   45.7703    1.5659   19.5034
4    Sc4_closed_combined  closed_loop    0.8500    0.8500    0.0207  312.5602  298.7130   91.5555    1.5600   26.6877
5   Sc5_closed_saturated  closed_loop    1.0000    0.4000    0.0153  314.1727  297.2247  382.9597    1.6032   22.7435
6    Sc6_open_with_fault    open_loop    1.0000    0.700

## Section 5 — Figure 3: `04_simulator_sanity.png` (SI S2)

Matched-protocol prior predictive check: per-draw `warm_start_ic` at each draw's own `(alpha, beta)` (the Item 8 fix), not a single fixed healthy warm start.


In [5]:
d = np.load(DATA / "observations.npz", allow_pickle=False)
x_all = jnp.asarray(d["x"]); t = jnp.asarray(d["t"]); sid = d["scenario_id"]; theta_all = d["theta"]
S_all = np.asarray(compute_summary_statistics_batch(x_all, t))
print(f"Summary statistics: {S_all.shape}  NaN={np.isnan(S_all).sum()}")

Y0_TRAIN = warm_start_ic(NOMINAL_PARAMS_CL, NOMINAL_INLET_CL, NOMINAL_CTRL)

N_PRIOR = 500
prior_check = box_uniform_2d()
theta_check = prior_check.sample((N_PRIOR,))
S_sim_raw = []
t0 = time.perf_counter()
for i in range(len(theta_check)):
    alpha_i, beta_i = float(theta_check[i, 0]), float(theta_check[i, 1])
    params_i = jnp.array([UA_NOMINAL, K0_NOMINAL, alpha_i, beta_i], dtype=jnp.float32)
    y0_i = warm_start_ic(params_i, NOMINAL_INLET_CL, NOMINAL_CTRL)
    proc_key, sens_key = jax.random.split(jax.random.PRNGKey(i))
    _, ys, qc = simulate_em_window(params_i, NOMINAL_INLET_CL, NOMINAL_CTRL, y0_i, key=proc_key)
    t_s = jnp.arange(1, ys.shape[0]+1) * 0.5
    obs_i = jnp.stack([ys[:,0], ys[:,1], ys[:,2], qc], axis=1)
    obs_i = apply_sensor_layer(obs_i, key=sens_key, noise_pct=DEFAULT_SENSOR_NOISE_PCT)
    S_sim_raw.append(np.asarray(compute_summary_statistics(obs_i, t_s)))
S_sim = np.stack(S_sim_raw)
print(f"{N_PRIOR} prior simulations: {time.perf_counter()-t0:.1f}s")

show = ["UA_eff_proxy", "k0_eff_proxy", "C_mean", "Qc_mean"]
feat_list = list(FEATURE_NAMES)
cl_scenarios = [1, 2, 3, 4, 5, 7]
cl_mask = np.isin(sid, cl_scenarios)
sc_colors = {1: "#ff7f0e", 2: "#d62728", 3: "#2ca02c", 4: "#9467bd", 5: "#8c564b", 7: "#e377c2"}
sc_labels = {1: "Sc1 healthy", 2: "Sc2 fouling", 3: "Sc3 decay", 4: "Sc4 combined", 5: "Sc5 saturated", 7: "Sc7 drift"}

fig, axes = plt.subplots(2, 4, figsize=(16, 7), constrained_layout=True)
for col, fname in enumerate(show):
    fi = feat_list.index(fname)
    sim_vals = S_sim[:, fi]
    sim_lo, sim_hi = np.percentile(sim_vals, [2.5, 97.5])

    ax = axes[0, col]
    xs = np.linspace(sim_vals.min() - 0.05 * np.ptp(sim_vals), sim_vals.max() + 0.05 * np.ptp(sim_vals), 300)
    kde = gaussian_kde(sim_vals)
    ax.fill_between(xs, kde(xs), alpha=0.3, color="steelblue", label="prior sim KDE")
    ax.axvspan(sim_lo, sim_hi, alpha=0.10, color="steelblue")
    for sc_id in cl_scenarios:
        obs_vals = S_all[sid == sc_id, fi]
        ax.plot(obs_vals, np.full_like(obs_vals, -0.02 * kde(xs).max()), '|',
                color=sc_colors[sc_id], markersize=8, label=sc_labels[sc_id])
    ax.set_title(fname, fontsize=9)
    if col == 0:
        ax.legend(fontsize=6, loc="upper left")
    ax.set_ylabel("density")

    ax = axes[1, col]
    ax.hist(sim_vals, bins=40, alpha=0.4, color="steelblue", density=True, label="prior sim")
    for sc_id in cl_scenarios:
        obs_vals = S_all[sid == sc_id, fi]
        ax.axvline(obs_vals.mean(), color=sc_colors[sc_id], lw=1.5, ls="--", label=f"{sc_labels[sc_id]} mean")
    ax.axvline(sim_lo, color="steelblue", lw=1, ls=":", alpha=0.7)
    ax.axvline(sim_hi, color="steelblue", lw=1, ls=":", alpha=0.7)
    ax.set_title(f"{fname} (hist + scenario means)", fontsize=8)
    ax.set_ylabel("density")
    if col == 0:
        ax.legend(fontsize=5, loc="upper left")

fig.suptitle(f"Prior predictive check: {N_PRIOR} prior simulations vs observed CL scenarios\n"
             "Blue shading = [2.5%, 97.5%] prior predictive interval", fontsize=11)
fig_path = FIGS / "04_simulator_sanity.png"
fig.savefig(fig_path, dpi=130, bbox_inches="tight")
plt.close(fig)
print(f"Saved: {fig_path}")

print("\nQuantitative coverage check (obs inside [2.5%, 97.5%] prior predictive):")
all_covered = True
coverage_rows = []
for fname in feat_list:
    fi = feat_list.index(fname)
    sim_lo, sim_hi = np.percentile(S_sim[:, fi], [2.5, 97.5])
    obs_cl = S_all[cl_mask, fi]
    inside = np.mean((obs_cl >= sim_lo) & (obs_cl <= sim_hi))
    if inside < 0.80:
        all_covered = False
    coverage_rows.append((fname, inside))
    flag = "" if inside >= 0.80 else "  <-- LOW"
    print(f"{fname:<20} {inside:>7.1%}{flag}")
print(f"\n{'PASSED' if all_covered else 'NEEDS REVIEW'} -- compare against SI S2's documented borderline features")


Summary statistics: (400, 29)  NaN=0


500 prior simulations: 306.7s


Saved: /Users/simo/inso-po-RD/cstr-model-optimisation/research_project/figures/04_simulator_sanity.png

Quantitative coverage check (obs inside [2.5%, 97.5%] prior predictive):
C_mean                100.0%
C_std                  99.3%
C_slope                99.0%
C_min                 100.0%
C_max                 100.0%
T_mean                 68.7%  <-- LOW
T_std                  94.0%
T_slope                96.7%
T_min                  83.0%
T_max                  85.7%
Tc_mean                97.3%
Tc_std                 96.7%
Tc_slope               97.3%
Tc_min                 95.7%
Tc_max                 97.3%
Qc_mean                97.0%
Qc_std                 97.7%
Qc_slope               97.7%
Qc_min                 97.0%
Qc_max                 97.7%
C_final_mean          100.0%
T_final_mean           75.0%  <-- LOW
Tc_final_mean          96.7%
Qc_final_mean          96.7%
int_abs_T_err          65.7%  <-- LOW
Qc_sat_low_frac        99.7%
Qc_sat_high_frac       97.3%
UA_eff_proxy 

## Section 6 — Figure 4: `03_pca.png` (SI S4)


In [6]:
scaler = StandardScaler().fit(S_all)
Z = scaler.transform(S_all)

pca = PCA(n_components=4).fit(Z)
Y_pca = pca.transform(Z)
print("explained variance ratio:", np.round(pca.explained_variance_ratio_, 3))

cmap = plt.get_cmap("tab10")
id_to_name = {sc.id: sc.name for sc in list_configs()}
mode_all = d["mode"]
fig, axes = plt.subplots(1, 2, figsize=(13, 5), constrained_layout=True)
for sc_id in sorted(np.unique(sid)):
    m = sid == sc_id
    color = cmap(int(sc_id) % 10)
    marker = "o" if mode_all[m][0] == b"closed_loop" or mode_all[m][0] == "closed_loop" else "s"
    axes[0].scatter(Y_pca[m, 0], Y_pca[m, 1], s=24, alpha=0.7, color=color, marker=marker, label=id_to_name[sc_id])
    axes[1].scatter(Y_pca[m, 2], Y_pca[m, 3], s=24, alpha=0.7, color=color, marker=marker)
axes[0].set(xlabel=f"PC1 ({pca.explained_variance_ratio_[0]*100:.1f} %)",
            ylabel=f"PC2 ({pca.explained_variance_ratio_[1]*100:.1f} %)", title="PCA: PC1-PC2")
axes[1].set(xlabel=f"PC3 ({pca.explained_variance_ratio_[2]*100:.1f} %)",
            ylabel=f"PC4 ({pca.explained_variance_ratio_[3]*100:.1f} %)", title="PCA: PC3-PC4")
axes[0].legend(fontsize=8, loc="best", ncol=2)
for ax in axes:
    ax.grid(alpha=0.3)
fig.suptitle("PCA on standardised 29-D summary statistics (circles: closed-loop, squares: open-loop)")
fig_path = FIGS / "03_pca.png"
fig.savefig(fig_path, dpi=140, bbox_inches="tight")
plt.close(fig)
print(f"Saved: {fig_path}")
print(f"PC1+PC2 variance explained: {100*(pca.explained_variance_ratio_[0]+pca.explained_variance_ratio_[1]):.1f}% "
      "-- compare against SI S4's documented 57.7% figure")


explained variance ratio: [0.462 0.115 0.081 0.07 ]


Saved: /Users/simo/inso-po-RD/cstr-model-optimisation/research_project/figures/03_pca.png
PC1+PC2 variance explained: 57.6% -- compare against SI S4's documented 57.7% figure


## Section 7 — Figure 5: `04_sensitivity_scatter.png` (SI S3)

**Cost/correctness tradeoff, matching this project's own established flagged-expensive-rerun convention** (`RUN_SENSITIVITY`-style flags throughout `notebooks/system_I/04_sbi_training.ipynb`):

- `RETRAIN_MATCHED_PROTOCOL = False` (default): reuses the existing cached `results/sbi_sensitivity.pkl`. **This cache predates the Item 8 matched-protocol warm-start fix** (generated 2026-06-07, uses a single fixed healthy warm start for every training draw) -- its absolute `beta_hat` values should be read as illustrating the *monotonic-convergence trend*, not as comparable to the corrected production posterior (see SI S3's own caveat text). Fast (seconds).
- `RETRAIN_MATCHED_PROTOCOL = True`: retrains all 3 budgets (1k/5k/10k draws, MAF) with `scenario_specific_warm_start=True`, closing the gap SI S3 flags as "not yet performed." **Expect roughly 1-2 hours** (the per-draw warm-start solve alone adds ~0.35s x 16,000 draws ≈ 1.5h, documented in `cstr_sbi.inference.simulation_wrapper_sbi`'s own docstring) -- do not flip this on without expecting to wait.


In [7]:
RETRAIN_MATCHED_PROTOCOL = False  # 2026-09-29: retrain already done and cached below; flip True only to redo it from scratch
MATCHED_CACHE_PATH = None  # set below once RESULTS is known

obs_sc1_idx = np.where(sid == 1)[0][0]
obs_sc2_idx = np.where(sid == 2)[0][0]
obs_sc1, alpha_sc1, beta_sc1 = S_all[obs_sc1_idx], float(theta_all[obs_sc1_idx, 2]), float(theta_all[obs_sc1_idx, 3])
obs_sc2, alpha_sc2, beta_sc2 = S_all[obs_sc2_idx], float(theta_all[obs_sc2_idx, 2]), float(theta_all[obs_sc2_idx, 3])
print(f"Sc1 true: alpha={alpha_sc1:.2f}  beta={beta_sc1:.2f}")
print(f"Sc2 true: alpha={alpha_sc2:.2f}  beta={beta_sc2:.2f}")

prior = box_uniform_2d()
sensitivity_results = []

if RETRAIN_MATCHED_PROTOCOL:
    for n_sims in [1_000, 5_000, 10_000]:
        print(f"\n--- Training (matched protocol): n_simulations={n_sims} (MAF, 128 hidden, 5 transforms) ---")
        posterior, meta = train_sbi_posterior(
            prior, n_simulations=n_sims, density_estimator="maf",
            hidden_features=128, num_transforms=5, training_batch_size=256,
            max_num_epochs=200, seed=42, y0=Y0_TRAIN,
            scenario_specific_warm_start=True,
        )
        s1 = sample_posterior(posterior, obs_sc1, n_samples=5_000)
        s2 = sample_posterior(posterior, obs_sc2, n_samples=5_000)
        m1 = summarise_posterior(s1, np.array([alpha_sc1, beta_sc1]))
        m2 = summarise_posterior(s2, np.array([alpha_sc2, beta_sc2]))
        sensitivity_results.append(dict(n_sims=n_sims, meta=meta, samples_sc1=s1, samples_sc2=s2, metrics_sc1=m1, metrics_sc2=m2))
        print(f"Sc2 beta: mean={s2[:,1].mean():.3f}  wall_time={meta['wall_time_s']:.0f}s")
    with open(RESULTS / "sbi_sensitivity_matched_protocol.pkl", "wb") as f:
        import pickle; pickle.dump(sensitivity_results, f)
    print("Saved sbi_sensitivity_matched_protocol.pkl")
else:
    import pickle
    matched_path = RESULTS / "sbi_sensitivity_matched_protocol.pkl"
    if matched_path.exists():
        with open(matched_path, "rb") as f:
            sensitivity_results = pickle.load(f)
        print(f"Loaded cached MATCHED-PROTOCOL sensitivity results from {matched_path.name}")
    else:
        with open(RESULTS / "sbi_sensitivity.pkl", "rb") as f:
            sensitivity_results = pickle.load(f)
        print("Loaded cached (pre-matched-protocol) sensitivity results -- see markdown caveat above")

fig, axes = plt.subplots(2, 3, figsize=(14, 7), constrained_layout=True)
for col, res in enumerate(sensitivity_results):
    n = res["n_sims"]
    for row, (sc_label, samp, alpha_t, beta_t) in enumerate([
        (f"Sc1 (alpha=1, beta=1)", res["samples_sc1"], alpha_sc1, beta_sc1),
        (f"Sc2 (alpha=1, beta=0.7)", res["samples_sc2"], alpha_sc2, beta_sc2),
    ]):
        ax = axes[row, col]
        ax.scatter(samp[:, 0], samp[:, 1], s=3, alpha=0.2, color="steelblue" if row == 0 else "tomato")
        ax.axvline(alpha_t, color="k", lw=1.5, ls="--", label=f"alpha={alpha_t:.2f}")
        ax.axhline(beta_t, color="purple", lw=1.5, ls="--", label=f"beta={beta_t:.2f}")
        ax.set(xlabel="alpha", ylabel="beta", title=f"{sc_label}\nn_sims={n:,}")
        ax.set_xlim(ALPHA_LOW - 0.02, ALPHA_HIGH + 0.02)
        ax.set_ylim(BETA_LOW - 0.02, BETA_HIGH + 0.02)
        ax.legend(fontsize=8); ax.grid(alpha=0.3)

protocol_tag = "matched protocol" if RETRAIN_MATCHED_PROTOCOL else "pre-matched-protocol cache"
fig.suptitle(f"SBI 2-D posterior [alpha, beta]: sensitivity over n_simulations (MAF, {protocol_tag})")
fig_path = FIGS / "04_sensitivity_scatter.png"
fig.savefig(fig_path, dpi=130, bbox_inches="tight")
plt.close(fig)
print(f"Saved: {fig_path}")

print("\nSensitivity summary:")
for res in sensitivity_results:
    s2 = res["samples_sc2"]
    print(f"  n_sims={res['n_sims']:>8,}  Sc2 beta_hat={s2[:,1].mean():.3f}")


Sc1 true: alpha=1.00  beta=1.00
Sc2 true: alpha=1.00  beta=0.70
Loaded cached MATCHED-PROTOCOL sensitivity results from sbi_sensitivity_matched_protocol.pkl


Saved: /Users/simo/inso-po-RD/cstr-model-optimisation/research_project/figures/04_sensitivity_scatter.png

Sensitivity summary:
  n_sims=   1,000  Sc2 beta_hat=0.747
  n_sims=   5,000  Sc2 beta_hat=0.735
  n_sims=  10,000  Sc2 beta_hat=0.733


## Section 8 — Figure 6: `04_sbc_ranks.png` (SI S6)

Loads the current production posterior (`results/sbi_posterior_final.pkl`, the matched-protocol posterior promoted 2026-08-17) and runs simulation-based calibration exactly as `notebooks/system_I/04_sbi_training.ipynb` does. **Note**: the 500 SBC test draws are not seeded here (matching the original notebook), so the exact KS p-values will differ slightly run to run -- update `supporting_information.tex` S6's reported numbers (currently 0.21/0.13) to whatever this run actually prints if they differ meaningfully, rather than leaving a mismatch between the figure and the prose.


In [8]:
import pickle
with open(RESULTS / "sbi_posterior_final.pkl", "rb") as f:
    d_final = pickle.load(f)
posterior_final, meta_final = d_final["posterior"], d_final["metadata"]
print(f"Loaded production posterior: {meta_final}")

from sbi.diagnostics import run_sbc, check_sbc

N_SBC = 500
torch.manual_seed(2026)  # 2026-09-29: seeded for reproducibility -- SBC p-values drifted run-to-run before this
theta_sbc = prior.sample((N_SBC,))

print(f"Simulating {N_SBC} SBC test cases (matched protocol, per-draw warm start)...")
t0 = time.perf_counter()
x_sbc_list = []
for i in range(N_SBC):
    x_i = simulation_wrapper_sbi(
        theta_sbc[i:i+1], inlet=NOMINAL_INLET_CL, ctrl=NOMINAL_CTRL, y0=Y0_TRAIN,
        seed=100_000 + i, scenario_specific_warm_start=True)
    x_sbc_list.append(x_i)
x_sbc = torch.cat(x_sbc_list)
print(f"Done in {time.perf_counter()-t0:.0f}s. Shapes: theta={theta_sbc.shape}, x={x_sbc.shape}")

print("Running SBC (1000 posterior samples per test case)...")
ranks, dap_samples = run_sbc(theta_sbc, x_sbc, posterior_final, num_posterior_samples=1000, reduce_fns="marginals")
checks = check_sbc(ranks, theta_sbc, dap_samples, num_posterior_samples=1000)
ks_alpha = float(checks['ks_pvals'][0])
ks_beta = float(checks['ks_pvals'][1])
c2st_alpha = float(checks['c2st_ranks'][0])
c2st_beta = float(checks['c2st_ranks'][1])

print(f"\nSBC Results:")
print(f"  KS p-values:  alpha = {ks_alpha:.4f},  beta = {ks_beta:.4f}")
print(f"  C2ST ranks:   alpha = {c2st_alpha:.3f},  beta = {c2st_beta:.3f}")
print(f"  KS p > 0.05 -> cannot reject uniformity (well-calibrated)")
print(f"  Compare against supporting_information.tex S6's reported 0.21/0.13 -- "
      f"update the .tex if these differ meaningfully (they will not be bit-identical, "
      f"since the 500 SBC draws are not seeded).")

fig, axes = plt.subplots(1, 2, figsize=(10, 4), constrained_layout=True)
for i, (ax, name, ks_p) in enumerate(zip(axes, ["alpha", "beta"], [ks_alpha, ks_beta])):
    ax.hist(ranks[:, i].numpy(), bins=20, density=True, alpha=0.7, color="steelblue", edgecolor="white")
    expected_density = 1.0 / (1001 / 20)
    ax.axhline(expected_density, color="k", ls="--", lw=1.5, label="uniform")
    ax.set_title(f"{name} rank histogram\nKS p = {ks_p:.4f} ({'pass' if ks_p > 0.05 else 'fail'})", fontsize=11)
    ax.set_xlabel("rank / (num_posterior_samples + 1)")
    ax.set_ylabel("density")
    ax.legend(); ax.grid(alpha=0.2)
fig.suptitle(f"SBC rank histograms ({N_SBC} test cases, 1000 posterior samples each)", fontsize=12)
fig_path = FIGS / "04_sbc_ranks.png"
fig.savefig(fig_path, dpi=130, bbox_inches="tight")
plt.close(fig)
print(f"Saved: {fig_path}")


Loaded production posterior: {'n_simulations': 10000, 'density_estimator': 'nsf', 'hidden_features': 128, 'num_transforms': 5, 'training_batch_size': 256, 'max_num_epochs': 200, 'wall_time_s': 4731.370576166984}
Simulating 500 SBC test cases (matched protocol, per-draw warm start)...


Done in 236s. Shapes: theta=torch.Size([500, 2]), x=torch.Size([500, 29])
Running SBC (1000 posterior samples per test case)...


/Users/simo/inso-po-RD/cstr-model-optimisation/research_project/.venv/lib/python3.10/site-packages/sbi/utils/diagnostics_utils.py:45: UserWarning: Capping max_sampling_batch_size from 10000 to 200 to avoid excessive memory usage.
  posterior_samples = posterior.sample_batched(


Drawing 1000 samples for 500 observations:   0%|          | 0/1000 [00:00<?, ?it/s]

/Users/simo/inso-po-RD/cstr-model-optimisation/research_project/.venv/lib/python3.10/site-packages/nflows/transforms/lu.py:80: UserWarning: torch.triangular_solve is deprecated in favor of torch.linalg.solve_triangularand will be removed in a future PyTorch release.
torch.linalg.solve_triangular has its arguments reversed and does not return a copy of one of the inputs.
X = torch.triangular_solve(B, A).solution
should be replaced with
X = torch.linalg.solve_triangular(A, B). (Triggered internally at /Users/runner/work/pytorch/pytorch/aten/src/ATen/native/BatchLinearAlgebra.cpp:2279.)
  outputs, _ = torch.triangular_solve(


Calculating ranks for 500 SBC samples:   0%|          | 0/500 [00:00<?, ?it/s]


SBC Results:
  KS p-values:  alpha = 0.3048,  beta = 0.7473
  C2ST ranks:   alpha = 0.559,  beta = 0.521
  KS p > 0.05 -> cannot reject uniformity (well-calibrated)
  Compare against supporting_information.tex S6's reported 0.21/0.13 -- update the .tex if these differ meaningfully (they will not be bit-identical, since the 500 SBC draws are not seeded).
Saved: /Users/simo/inso-po-RD/cstr-model-optimisation/research_project/figures/04_sbc_ranks.png


## Section 9 — Summary


In [9]:
print("=== All System I supplementary figures regenerated ===")
for fname in ["02_scenarios_overview.png", "02_headline_summary.png", "04_simulator_sanity.png",
              "03_pca.png", "04_sensitivity_scatter.png", "04_sbc_ranks.png"]:
    p = FIGS / fname
    print(f"  {'OK' if p.exists() else 'MISSING':8s} {p}")
print()
print("Next: re-run pdflatex/latexmk on docs/Manuscript/supporting_information.tex")
print("and check S6's reported KS p-values against this run's printed output.")


=== All System I supplementary figures regenerated ===
  OK       /Users/simo/inso-po-RD/cstr-model-optimisation/research_project/figures/02_scenarios_overview.png
  OK       /Users/simo/inso-po-RD/cstr-model-optimisation/research_project/figures/02_headline_summary.png
  OK       /Users/simo/inso-po-RD/cstr-model-optimisation/research_project/figures/04_simulator_sanity.png
  OK       /Users/simo/inso-po-RD/cstr-model-optimisation/research_project/figures/03_pca.png
  OK       /Users/simo/inso-po-RD/cstr-model-optimisation/research_project/figures/04_sensitivity_scatter.png
  OK       /Users/simo/inso-po-RD/cstr-model-optimisation/research_project/figures/04_sbc_ranks.png

Next: re-run pdflatex/latexmk on docs/Manuscript/supporting_information.tex
and check S6's reported KS p-values against this run's printed output.
